# Encoding Categorical Data

Machine learning models work with numbers, but many columns hold text, like city, education level, or color. Encoding means turning those text categories into numbers.

Categorical data comes in two kinds:

* **Nominal data** has no order. City, color, and gender are examples. Karachi is not "greater" than Peshawar.
* **Ordinal data** has a natural order. Education level (school, college, university) and size (small, medium, large) are examples.

The kind of data decides which encoding we use:

* **Ordinal encoding** is for ordinal data. We give the order ourselves, so the numbers keep the ranking.
* **Label encoding** is for the target column only. It gives each category a number, based on alphabetical order.
* **One hot encoding** is for nominal data. It makes a separate 0/1 column for each category, so no false ranking is created.

In [1]:
import pandas as pd

df = pd.DataFrame({
    'city': ['Peshawar', 'Lahore', 'Karachi', 'Lahore', 'Peshawar'],
    'education': ['school', 'university', 'college', 'college', 'university'],
    'bought': ['no', 'yes', 'yes', 'no', 'yes']
})

df

,city,education,bought
0,Peshawar,school,no
1,Lahore,university,yes
2,Karachi,college,yes
3,Lahore,college,no
4,Peshawar,university,yes


In this table:

* `city` is nominal, there is no order between the cities.
* `education` is ordinal, school < college < university.
* `bought` is the target column, the thing we want to predict.

We will encode each one with the method that fits it, starting with `education`.

## Ordinal Encoding

Ordinal encoding replaces each category with a number, and we choose the order ourselves. For education, the order is school, then college, then university, so they become 0, 1, 2.

This keeps the ranking, so the model can see that university is "more" than school.

In [5]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[['school', 'college', 'university']], dtype='int')
df['encoded_edu'] = oe.fit_transform(df[['education']])
df

,city,education,bought,encoded_edu
0,Peshawar,school,no,0
1,Lahore,university,yes,2
2,Karachi,college,yes,1
3,Lahore,college,no,1
4,Peshawar,university,yes,2


Because we gave the order, school became 0, college 1, and university 2. If we had not passed `categories`, sklearn would sort the words alphabetically (college, school, university) as below, which would give the wrong ranking. That is why the order matters for ordinal data.

In [8]:
default_encoder = OrdinalEncoder(dtype='int')
df['education_default'] = default_encoder.fit_transform(df[['education']])

df

,city,education,bought,encoded_edu,education_default
0,Peshawar,school,no,0,1
1,Lahore,university,yes,2,2
2,Karachi,college,yes,1,0
3,Lahore,college,no,1,0
4,Peshawar,university,yes,2,2


## Label Encoding

Label encoding gives each category a number, starting from 0, in alphabetical order. For our `bought` column, "no" becomes 0 and "yes" becomes 1.

Unlike ordinal encoding, we do not choose the order. sklearn sorts the labels alphabetically and numbers them. That is fine here because the target column is the thing we want to predict, so the numbers are just names for the classes, not a ranking.

This is why label encoding is meant for the target column only. If we used it on an input column like city, the model would read the numbers as a ranking (Karachi = 0, Lahore = 1, Peshawar = 2), and that ranking means nothing.


In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['bought_encoded'] = label_encoder.fit_transform(df['bought'])

df

,city,education,bought,encoded_edu,education_default,bought_encoded
0,Peshawar,school,no,0,1,0
1,Lahore,university,yes,2,2,1
2,Karachi,college,yes,1,0,1
3,Lahore,college,no,1,0,0
4,Peshawar,university,yes,2,2,1


In [10]:
# which number belongs to which label
print(label_encoder.classes_)

# going back from numbers to the original labels
print(label_encoder.inverse_transform([0, 1, 1, 0]))

['no' 'yes']
['no' 'yes' 'yes' 'no']


`classes_` shows the order sklearn used, and the position in that list is the number each label got. `inverse_transform` goes the other way, turning numbers back into the original labels. This is useful after a model predicts 0 or 1 and we want to show "no" or "yes" again.

## One Hot Encoding

One hot encoding is for nominal data, where the categories have no order. Instead of giving each city a number, we make a separate column for every city. In each row, the column for that row's city gets a 1 and the others get a 0.

This avoids the problem we saw earlier. If we gave Karachi 0, Lahore 1 and Peshawar 2, the model would think Peshawar is "bigger" than Karachi. With separate 0/1 columns, no city is ranked above another.

For our `city` column, we get three new columns: `city_Karachi`, `city_Lahore` and `city_Peshawar`.

In [14]:
df

,city,education,bought,encoded_edu,education_default,bought_encoded
0,Peshawar,school,no,0,1,0
1,Lahore,university,yes,2,2,1
2,Karachi,college,yes,1,0,1
3,Lahore,college,no,1,0,0
4,Peshawar,university,yes,2,2,1


In [16]:
pd.get_dummies(df['city'], dtype=int)

,Karachi,Lahore,Peshawar
0,0,0,1
1,0,1,0
2,1,0,0
3,0,1,0
4,0,0,1


This is the plain pandas way, and it shows the idea clearly. Row 0 is Peshawar, so only the `Peshawar` column has a 1. In sklearn we use `OneHotEncoder`, which does the same job but remembers what it learned from the training data, so it can be reused on test data later.

In [13]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, dtype=int)
city_encoded = ohe.fit_transform(df[['city']])

city_df = pd.DataFrame(city_encoded, columns=ohe.get_feature_names_out())
city_df

,city_Karachi,city_Lahore,city_Peshawar
0,0,0,1
1,0,1,0
2,1,0,0
3,0,1,0
4,0,0,1


In [18]:
ohe.inverse_transform(city_df)

array([['Peshawar'],
       ['Lahore'],
       ['Karachi'],
       ['Lahore'],
       ['Peshawar']], dtype=object)

## The Dummy Variable Trap and drop='first'

Look at our one hot columns again. Each row has exactly one 1, so the three city columns always add up to 1. That means if we know two of the columns, we can work out the third one. If `city_Karachi = 0` and `city_Lahore = 0`, the city must be Peshawar.

So the third column carries no new information. It is a copy of what the other two already tell us. In linear models like linear and logistic regression, this causes a problem called **multicollinearity**: one column can be predicted perfectly from the others, so the model cannot decide how to split the weight between them. This is known as the **dummy variable trap**.

The fix is to drop one column. With `drop='first'`, sklearn removes the first category's column (Karachi here). That category is not lost. It becomes the baseline, and it is shown by all the other columns being 0.

* Karachi: `[0, 0]`
* Lahore: `[1, 0]`
* Peshawar: `[0, 1]`

We need n - 1 columns to describe n categories.

In [19]:
ohe_drop = OneHotEncoder(sparse_output=False, dtype=int, drop='first')
city_dropped = ohe_drop.fit_transform(df[['city']])

city_dropped_df = pd.DataFrame(city_dropped, columns=ohe_drop.get_feature_names_out())
city_dropped_df

,city_Lahore,city_Peshawar
0,0,1
1,1,0
2,0,0
3,1,0
4,0,1


In [23]:
print("Row sums with all 3 columns:", city_df.sum(axis=1).tolist())

print("Row sums after drop='first':", city_dropped_df.sum(axis=1).tolist())

Row sums with all 3 columns: [1, 1, 1, 1, 1]
Row sums after drop='first': [1, 1, 0, 1, 1]


The first line shows that every row sums to exactly 1 when we keep all three columns. That is the redundancy. After `drop='first'`, the sums are no longer all 1, because the dropped city is now represented by zeros.

When to use it: linear models (linear regression, logistic regression) need `drop='first'`. Tree based models like decision trees and random forests do not suffer from this, so keeping all columns is fine there. When we do not know which model we will use later, dropping one column is the safer habit.

In [35]:
test_Cat = [[0, 0], [1, 0], [0, 1]]

ohe_drop.inverse_transform(test_Cat)

array([['Karachi'],
       ['Lahore'],
       ['Peshawar']], dtype=object)

With `drop='first'`, a row of all zeros points back to the dropped category, so `[0, 0]` becomes Karachi.